# DLGenAI Project — Milestone 5
**Roll No:** 23f3004491

Ensembling, weighted averaging, TTA, and MAP@3 with two fine-tuned sequence
classification checkpoints (DeBERTa-v3-small + RoBERTa-base, 5 labels = A-E).
Each answer cell prints `Q# answer:`.

In [1]:
!pip install -q -U "transformers==4.46.3" "accelerate==1.1.1"
print("ready")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 73.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.2/333.2 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 100.5 MB/s eta 0:00:00
ready


In [2]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch

BASE = "/kaggle/input/competitions/smart-mcq-solver-challenge"
train = pd.read_csv(f"{BASE}/train.csv")
test  = pd.read_csv(f"{BASE}/test.csv")

OPTIONS = ['A', 'B', 'C', 'D', 'E']
label_map = {o: i for i, o in enumerate(OPTIONS)}
train['label'] = train['answer'].map(label_map)

print(train.shape, test.shape)

(2000, 9) (500, 7)


## Setup — fine-tune the two checkpoints

Both models are 5-label sequence classifiers: input = prompt + all five options in one
sequence, label = index of the correct option (A=0 ... E=4).

In [3]:
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer)

SEED = 42
tr_df, val_df = train_test_split(train, test_size=0.2, random_state=SEED, shuffle=True)
tr_df, val_df = tr_df.reset_index(drop=True), val_df.reset_index(drop=True)

def build_text(row):
    opts = " ".join(f"({o}) {row[o]}" for o in OPTIONS)
    return f"{row['prompt']} {opts}"

def make_ds(df, tokenizer, with_labels=True):
    texts = [build_text(r) for _, r in df.iterrows()]
    enc = tokenizer(texts, truncation=True, max_length=384, padding='max_length')
    data = {'input_ids': enc['input_ids'], 'attention_mask': enc['attention_mask']}
    if with_labels:
        data['labels'] = df['label'].tolist()
    ds = Dataset.from_dict(data)
    ds.set_format('torch')
    return ds

def finetune(model_name, run_dir):
    tok = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=5)
    args = TrainingArguments(
        output_dir=run_dir,
        num_train_epochs=2,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=16,
        learning_rate=2e-5,
        eval_strategy="epoch",
        save_strategy="no",
        logging_steps=50,
        report_to="none",
        fp16=True,
        seed=SEED,
    )
    trainer = Trainer(model=model, args=args,
                      train_dataset=make_ds(tr_df, tok),
                      eval_dataset=make_ds(val_df, tok),
                      processing_class=tok)
    trainer.train()
    return trainer, tok

deberta_trainer, deberta_tok = finetune("microsoft/deberta-v3-small", "./m5_deberta")
roberta_trainer, roberta_tok = finetune("roberta-base", "./m5_roberta")
print("Both checkpoints fine-tuned.")

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/286M [00:00<?, ?B/s]

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-small and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss
1,1.263800,0.842298
2,0.316000,0.202662


Both checkpoints fine-tuned.


In [4]:
@torch.no_grad()
def get_probs(trainer, tok, df, prefix=""):
    """Softmax class probabilities for every row of df, optional prompt prefix."""
    df2 = df.copy()
    if prefix:
        df2['prompt'] = prefix + " " + df2['prompt'].astype(str)
    ds = make_ds(df2, tok, with_labels=False)
    logits = trainer.predict(ds).predictions
    return torch.softmax(torch.tensor(logits).float(), dim=1).numpy()

deberta_test = get_probs(deberta_trainer, deberta_tok, test)
roberta_test = get_probs(roberta_trainer, roberta_tok, test)
print("Test probabilities computed:", deberta_test.shape, roberta_test.shape)

Test probabilities computed: (500, 5) (500, 5)


## Q1. DeBERTa top option + probability (test row index 25)

In [5]:
p25_d = deberta_test[25]
top_d = int(np.argmax(p25_d))

print("DeBERTa probs row 25:", {o: round(float(p), 4) for o, p in zip(OPTIONS, p25_d)})
print(f"Q1 answer: {OPTIONS[top_d]}, {p25_d[top_d]:.4f}")

DeBERTa probs row 25: {'A': 0.2718, 'B': 0.0397, 'C': 0.1281, 'D': 0.1924, 'E': 0.368}
Q1 answer: E, 0.3680


## Q2. Simple average ensemble — top option (row 25)

In [6]:
p25_r = roberta_test[25]
p25_avg = (p25_d + p25_r) / 2

print("Averaged probs:", {o: round(float(p), 4) for o, p in zip(OPTIONS, p25_avg)})
print("Q2 answer:", OPTIONS[int(np.argmax(p25_avg))])

Averaged probs: {'A': 0.1667, 'B': 0.0316, 'C': 0.1088, 'D': 0.1215, 'E': 0.5714}
Q2 answer: E


## Q3. Weighted ensemble 0.7 DeBERTa + 0.3 RoBERTa — top option (row 25)

In [7]:
p25_w = 0.7 * p25_d + 0.3 * p25_r

print("Weighted probs:", {o: round(float(p), 4) for o, p in zip(OPTIONS, p25_w)})
print("Q3 answer:", OPTIONS[int(np.argmax(p25_w))])

Weighted probs: {'A': 0.2088, 'B': 0.0348, 'C': 0.1165, 'D': 0.1498, 'E': 0.49}
Q3 answer: E


## Q4. Top-3 prediction string for row 25 (Kaggle format)

In [8]:
order25 = np.argsort(p25_w)[::-1]
top3_str = " ".join(OPTIONS[i] for i in order25[:3])

print("Q4 answer:", top3_str)

Q4 answer: E A D


## Q5. Weighted ensemble on all of test.csv → submission.csv row count

In [9]:
ens_test = 0.7 * deberta_test + 0.3 * roberta_test
orders = np.argsort(-ens_test, axis=1)
preds = [" ".join(OPTIONS[i] for i in row[:3]) for row in orders]

submission = pd.DataFrame({"id": test['id'], "prediction": preds})
submission.to_csv("submission.csv", index=False)

print("Q5 answer:", len(submission))

Q5 answer: 500


## Q6. Test-Time Augmentation (first 50 rows, DeBERTa)

Two passes: original prompt vs instruction-prefixed prompt; average probabilities;
count rows whose Top-1 changes vs the original-only prediction.

In [10]:
first50 = test.head(50).reset_index(drop=True)

probs_orig = deberta_test[:50]
probs_aug  = get_probs(deberta_trainer, deberta_tok, first50,
                       prefix="Answer the following multiple-choice question carefully:")
probs_tta  = (probs_orig + probs_aug) / 2

top1_orig = np.argmax(probs_orig, axis=1)
top1_tta  = np.argmax(probs_tta, axis=1)
changed = int((top1_orig != top1_tta).sum())

print("Q6 answer:", changed)

Q6 answer: 1


## Q7. DeBERTa vs weighted ensemble — Top-1 disagreements (first 100 rows)

In [11]:
d100 = np.argmax(deberta_test[:100], axis=1)
e100 = np.argmax(ens_test[:100], axis=1)

print("Q7 answer:", int((d100 != e100).sum()))

Q7 answer: 4


## Q8. Positive confidence gain rows (first 100)

In [12]:
conf_d = deberta_test[:100].max(axis=1)
conf_e = ens_test[:100].max(axis=1)
gain = conf_e - conf_d

print("Q8 answer:", int((gain > 0).sum()))

Q8 answer: 93


## Q9. Top-3 ordering changes after ensembling (first 100)

In [13]:
d_top3 = [tuple(np.argsort(-deberta_test[i])[:3]) for i in range(100)]
e_top3 = [tuple(np.argsort(-ens_test[i])[:3]) for i in range(100)]
diff = sum(1 for a, b in zip(d_top3, e_top3) if a != b)

print("Q9 answer:", diff)

Q9 answer: 30


## Q10. MAP@3 of the weighted ensemble on the first 100 validation samples

In [14]:
def map_at_3(true_idx, prob_matrix):
    total = 0.0
    for t, probs in zip(true_idx, prob_matrix):
        order = np.argsort(-probs)[:3]
        for rank, o in enumerate(order):
            if o == t:
                total += 1.0 / (rank + 1)
                break
    return total / len(true_idx)

val100 = val_df.head(100).reset_index(drop=True)
d_val = get_probs(deberta_trainer, deberta_tok, val100)
r_val = get_probs(roberta_trainer, roberta_tok, val100)
ens_val = 0.7 * d_val + 0.3 * r_val

score = map_at_3(val100['label'].tolist(), ens_val)
print("Q10 answer:", round(score, 4))

Q10 answer: 0.995
